[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Saving a Model


## What you will be able to do

Save a fitted pipeline to a file, load it in another Python program, and record beside it the
versions it needs, so that the predictions it makes later are the ones you tested.

## The idea

### The problem

Every model so far in this guide has lived in the memory of the notebook that fitted it. When the
runtime restarts, the model is gone, and getting it back means running every cell again with the
same data and the same choices. A job that predicts tomorrow's frost risk for 1,200 stations every
evening, or a web service that answers for a planned station, is a separate program, which should
not have to train anything and needs the model you tested, exactly as you tested it.

The obvious answer is to write the model to a file. The less obvious part is what such a file holds.
A table of numbers saved as CSV can be read by any program in any language. A saved scikit-learn
model can be read only by Python, only with scikit-learn installed, and safely only with the
scikit-learn version that wrote it, and opening one runs code.

### What saving a model means

> To **persist** a model is to write the fitted object to a file and rebuild it later.
> **`joblib.dump(model, path)`** writes it in a file format built on Python's **pickle**, and
> **`joblib.load(path)`** rebuilds it with pickle's loader. The file holds the model's settings,
> what it learned, and the names of the classes and functions it is made of, and not their code, so
> loading needs those names to exist, in versions that agree with the ones that saved it.

### Why it works that way

A pickle of a pipeline says, in effect, "a Pipeline whose first step is a StandardScaler with these
means and these scales, and whose second is a LinearRegression with these coefficients". Loading it
imports `StandardScaler` and `LinearRegression` from the scikit-learn that is installed now, makes
empty objects of those classes, and pours the saved values into them.

The classes must match. If a later scikit-learn changed what a class stores, the saved values may
no longer fit it, so scikit-learn writes its own version into the file and warns when the version
that loads it differs. The fix is to record the versions beside the model and install the same ones
wherever it is loaded, which is the version pin this notebook writes in its worked examples.

Your own functions are saved by name only. A pipeline step that calls a function you wrote in the
notebook loads only in a program where a function of that name exists, so such a function belongs in
a module that both programs import.

Loading a pickle runs whatever the file tells Python to call. A file from someone else can do
anything on your machine that you could, so load only files you made or files from a source you
trust as much as your own code.

Save the whole pipeline, never the final model alone. A scaler's means and an encoder's categories
are as much a part of what was learned as the coefficients.

### Where this shows up

Saving matters wherever a model is used outside the notebook that trained it, such as a scheduled
batch job, a dashboard, or a web service like the ones the **APIs and JSON** guide builds with
FastAPI. The **A Complete Workflow** notebook ends by saving its model this way.

### What this notebook covers

- saving a fitted pipeline with `joblib.dump` and loading it with `joblib.load`
- the scikit-learn version the file records, and writing every version down beside the model
- loading the model in a Python of its own, which never saw the training data
- a version mismatch, a missing function, a model saved without its scaler, and a file that runs
  code

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import joblib
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit([[1.0], [2.0], [3.0]], [2.0, 4.1, 5.9])
joblib.dump(model, "model.joblib")
again = joblib.load("model.joblib")
print(again.predict([[4.0]]).round(2), again is model)
```

```
[7.9] False
```

`joblib.dump` wrote the fitted model to a file, and `joblib.load` built a new object from it, which
predicts 7.9 for 4.0. `again is model` is `False` because it is a separate copy, made from the file,
and the worked examples check that its predictions match the original's.

## Setup

Seven tools from the standard library, the data libraries, and the parts of scikit-learn this
notebook saves and loads.

- `joblib` saves and loads models, and the cell prints its version beside scikit-learn's
- `json` writes the versions and the features down beside the model, and reads them back
- `subprocess` and `sys` run a separate Python that loads the model, and `sys.executable` names the
  Python that is running this notebook
- `pickle` writes the file that runs code as it loads, in Common errors
- `warnings` catches the warnings that loading prints, and `InconsistentVersionWarning` is the one
  for a file saved by another version, which Common errors makes by changing
  `sklearn.base.__version__` for a moment
- `Path` names the files, and `shutil` removes the scratch folder at the end
- `numpy` and `pandas` build the stations and compare predictions
- `make_pipeline`, `StandardScaler`, `LinearRegression` and `train_test_split` make the model to
  save, `RandomForestRegressor` the one whose version is changed, `LogisticRegression` the one saved
  without its scaler, and `FunctionTransformer` the step that calls a function of your own

`station_network` makes the network of 1,200 weather stations in Norway. The stations are made up,
and a generator started from a fixed seed draws them, so the table is the same on every run.
`FEATURES` names the five numeric columns, and `SCRATCH` is a folder for the files this notebook
writes.

In [1]:
import json
import pickle
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print(f"scikit-learn {sklearn.__version__} | joblib {joblib.__version__} | {len(network)} stations")

scikit-learn 1.6.1 | joblib 1.6.0 | 1200 stations


## Worked examples

### The pipeline to save

The model to save scales the five features and fits a line, trained on 80% of the stations and
scored on the rest:

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    network[FEATURES], network["jan_mean"], test_size=0.2, random_state=0
)
model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)

print("R² on the test stations:", round(model.score(X_test, y_test), 3))

R² on the test stations: 0.897


### Saving it to a file

`joblib.dump` takes the object and a path, and returns the names of the files it wrote:

In [3]:
path = SCRATCH / "jan_model.joblib"

print(joblib.dump(model, path))
print(sorted(item.name for item in SCRATCH.iterdir()))

['scratch/jan_model.joblib']
['jan_model.joblib']


One file holds the whole pipeline, the scaler's means and scales as well as the coefficients. The
`.joblib` ending is a convention that says which library wrote it, and joblib reads the file
whatever it is called.

### Loading it back

In [4]:
loaded = joblib.load(path)

print(type(loaded).__name__, [name for name, step in loaded.steps])
print("same predictions:", np.array_equal(loaded.predict(X_test), model.predict(X_test)))
print("same object:", loaded is model)

Pipeline ['standardscaler', 'linearregression']
same predictions: True
same object: False


The loaded pipeline is a new object built from the file, and it predicts exactly what the original
does for every test station. Nothing was fitted again, so it needed neither the training rows nor
the code that chose them.

### The version inside the file

What scikit-learn writes into a pickle comes from an object's `__getstate__`, and among the values
it adds the version of scikit-learn that saved it:

In [5]:
print("saved by:", model.__getstate__()["_sklearn_version"])
print("each step:", [step.__getstate__()["_sklearn_version"] for name, step in model.steps])
print("running:", sklearn.__version__)

saved by: 1.6.1
each step: ['1.6.1', '1.6.1']
running: 1.6.1


Each step records it separately, because each is loaded separately. Loading compares the recorded
version with the running one, and Common errors shows what happens when they differ.

### Writing the versions down beside the model

The version inside the file is checked only when the file is loaded, which is too late to install
the right one. A small JSON file beside the model says, before anything is loaded, what it needs and
what it was tested on:

In [6]:
about = {
    "model": path.name,
    "target": "jan_mean",
    "features": FEATURES,
    "test_r2": round(model.score(X_test, y_test), 3),
    "versions": {
        "scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__,
        "joblib": joblib.__version__,
    },
}
(SCRATCH / "jan_model.json").write_text(json.dumps(about, indent=2))

print(json.loads((SCRATCH / "jan_model.json").read_text())["versions"])

{'scikit-learn': '1.6.1', 'numpy': '2.1.3', 'pandas': '2.2.3', 'joblib': '1.6.0'}


The same versions, written as pins, are what a program that loads the model installs first:

In [7]:
pins = "\n".join(f"{name}=={version}" for name, version in about["versions"].items())
(SCRATCH / "requirements.txt").write_text(pins + "\n")

print((SCRATCH / "requirements.txt").read_text())

scikit-learn==1.6.1
numpy==2.1.3
pandas==2.2.3
joblib==1.6.0



`pip install -r requirements.txt` is how a program that loads the model gets these versions. NumPy,
pandas and joblib are pinned too, because joblib reads the file, and the arrays inside it and the
table handed to `predict` come from NumPy and pandas.

### Loading it in a Python of its own

The real test of a saved model is a program that never ran this notebook. `subprocess.run` starts a
second Python, which loads the file and predicts the January mean of a planned station.
`sys.executable` is the Python running this notebook, so the second program has the same packages
installed and none of this notebook's variables:

In [8]:
loader = '''
import sys
import joblib
import pandas as pd

model = joblib.load("scratch/jan_model.joblib")
planned = pd.DataFrame([[69.5, 23.0, 450, 21.0, 2026]],
                       columns=["latitude", "longitude", "elevation_m", "coast_km", "installed"])
print("the other Python predicted:", model.predict(planned).round(1))
print("scikit-learn imported by the load:", "sklearn" in sys.modules)
'''
run = subprocess.run([sys.executable, "-c", loader], capture_output=True, text=True)

print("exit code:", run.returncode)
print(run.stdout.strip())

exit code: 0
the other Python predicted: [-8.5]
scikit-learn imported by the load: True


The second program's own code imports only joblib and pandas. Loading the file imported the
scikit-learn classes it names as well, from the same installation as this notebook. Here is the same
prediction from the model in this notebook, for comparison:

In [9]:
planned = pd.DataFrame([[69.5, 23.0, 450, 21.0, 2026]], columns=FEATURES)

print(model.predict(planned).round(1))

[-8.5]


## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/13-saving-a-model-solutions.ipynb).

**1.** Fit `RandomForestRegressor(n_estimators=50, random_state=0)` on `X_train` and `y_train`, save it to `scratch/forest.joblib`, load it back, and check that its predictions for `X_test` are the same as the original's.

In [10]:
# your code here

**2.** Save `model` again with `joblib.dump(model, SCRATCH / "small.joblib", compress=3)`, load that file, and check that its predictions match the uncompressed one's.

In [11]:
# your code here

**3.** Read `scratch/jan_model.json` back with `json.loads`, and print the scikit-learn version it records beside `sklearn.__version__`, with whether they are equal.

In [12]:
# your code here

**4.** Make loading a file from another scikit-learn version fail instead of warn. Save a forest under the version `1.5.2`, as Common errors does, then load it inside `warnings.catch_warnings()` after calling `warnings.simplefilter("error", InconsistentVersionWarning)`. The warning is raised as an exception of its own class, so catch it with `except InconsistentVersionWarning as error`.

In [13]:
# your code here

## Common errors

### InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.5.2 when using version 1.6.1.

A file saved by scikit-learn 1.5.2 and loaded by 1.6.1 warns this way. This session has only 1.6.1,
so the cell sets `sklearn.base.__version__`, the version scikit-learn writes into every object it
saves, to `1.5.2` while it saves a forest, and puts it back before loading the file:

In [14]:
forest = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_train, y_train)

running = sklearn.base.__version__
sklearn.base.__version__ = "1.5.2"
joblib.dump(forest, SCRATCH / "forest_1_5_2.joblib")
sklearn.base.__version__ = running

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    older = joblib.load(SCRATCH / "forest_1_5_2.joblib")

version_warnings = [w for w in caught if w.category is InconsistentVersionWarning]
names = pd.Series([w.message.estimator_name for w in version_warnings])
print(names.value_counts().sort_index())
print("trees in the forest:", len(forest.estimators_))
first = caught[0]
print(f"{first.category.__name__}: {str(first.message).split(' This might')[0]}")
print("same predictions here:", np.array_equal(older.predict(X_test), forest.predict(X_test)))

DecisionTreeRegressor    51
RandomForestRegressor     1
Name: count, dtype: int64
trees in the forest: 50
same predictions here: True


Every object in the file that scikit-learn made warns once: the forest, its 50 trees, and the
unfitted tree it makes them from, which is why `DecisionTreeRegressor` has 51 warnings and comes
first. The rest of each warning says the mismatch "might lead to breaking code or invalid results".
The predictions match here because the file really came from 1.6.1, and only its label was changed.
A real file from 1.5.2 carries no such promise. Install the versions the model's JSON file records,
with the pins written beside it, or fit the model again under the new version and save it again.

### AttributeError: Can't get attribute 'add_features' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>

`add_features` is a function written in this notebook, used as a pipeline step through
`FunctionTransformer`. The pipeline saves without complaint, and a second Python fails to load it:

In [15]:
def add_features(table):
    """Latitude times elevation in kilometers, a feature for high stations far north."""
    return table.assign(north_and_high=table["latitude"] * table["elevation_m"] / 1000)


custom = make_pipeline(FunctionTransformer(add_features), StandardScaler(), LinearRegression())
joblib.dump(custom.fit(X_train, y_train), SCRATCH / "custom.joblib")

run = subprocess.run([sys.executable, "-c", "import joblib; joblib.load('scratch/custom.joblib')"],
                     capture_output=True, text=True)
print("exit code:", run.returncode)
print(run.stderr.strip().splitlines()[-1])

exit code: 1
AttributeError: Can't get attribute 'add_features' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


The file recorded the function as `__main__.add_features`, its name in the notebook, and the second
Python has no function of that name. The error is printed here because it happened in the other
program, and this cell only reads what that program wrote. The fix is to put the function in a
module, a `.py` file that both programs import, so the file records it under the module's name:

In [16]:
(SCRATCH / "station_features.py").write_text(
    "def add_features(table):\n"
    '    """Latitude times elevation in kilometers, a feature for high stations far north."""\n'
    '    return table.assign(north_and_high=table["latitude"] * table["elevation_m"] / 1000)\n'
)
sys.path.insert(0, str(SCRATCH))
import station_features

custom = make_pipeline(FunctionTransformer(station_features.add_features), StandardScaler(),
                       LinearRegression())
joblib.dump(custom.fit(X_train, y_train), SCRATCH / "custom.joblib")

loader = ("import sys, joblib; sys.path.insert(0, 'scratch'); "
          "joblib.load('scratch/custom.joblib'); print('loaded')")
run = subprocess.run([sys.executable, "-c", loader], capture_output=True, text=True)
print("exit code:", run.returncode, "|", run.stdout.strip())

exit code: 0 | loaded


`sys.path.insert(0, ...)` lets both programs find the module in the scratch folder. In a project,
the module would sit with the rest of the code, where every program that loads the model can import
it.

### No error, and a model saved without its scaler predicts snow for most stations

Here a logistic regression learns snow on 1 May from scaled features, and only the model is saved,
without the scaler:

In [17]:
snow = network["snow_on_may_1"]
X_tr, X_te, s_tr, s_te = train_test_split(network[FEATURES], snow, test_size=0.2, random_state=0,
                                          stratify=snow)
scaler = StandardScaler().fit(X_tr)
classifier = LogisticRegression().fit(scaler.transform(X_tr), s_tr)
joblib.dump(classifier, SCRATCH / "snow_model.joblib")

alone = joblib.load(SCRATCH / "snow_model.joblib")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    raw_answers = alone.predict(X_te)

print("predicted to have snow:", round(raw_answers.mean(), 3), "| measured:", round(s_te.mean(), 3))
print("accuracy given raw features:", round((raw_answers == s_te).mean(), 3))
print("accuracy given scaled ones:", round(alone.score(scaler.transform(X_te), s_te), 3))
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")

predicted to have snow: 0.912 | measured: 0.388
accuracy given raw features: 0.475
accuracy given scaled ones: 0.846


The loaded model was given the stations' features as they are, years like 1987 and elevations in the
hundreds, where it had learned from values a few units either side of zero. It predicted snow for
most stations and was right less than half the time. The warning is the only sign, and it says only
that the columns had names this time. Save the pipeline, scaler and model together, as the worked
examples did, and the program that loads it cannot leave a step out.

### No error, and a line that ran while a file was loading

A pickle can tell Python to call any function while it loads. This one was written by a class whose
`__reduce__` names `print`, which is harmless, and a file from someone else could name anything:

In [18]:
class Surprise:
    def __reduce__(self):
        return (print, ("this line ran while the file was loading",))


with open(SCRATCH / "surprise.pkl", "wb") as file:
    pickle.dump(Surprise(), file)

print("loading with pickle...")
with open(SCRATCH / "surprise.pkl", "rb") as file:
    result = pickle.load(file)
print("what came back:", result)

print("loading with joblib...")
print("what came back:", joblib.load(SCRATCH / "surprise.pkl"))

loading with pickle...
this line ran while the file was loading
what came back: None
loading with joblib...
this line ran while the file was loading
what came back: None


Loading the file ran `print` before anything came back, and what came back is `print`'s return
value, `None`. `joblib.load` runs the file the same way, as the second load shows, so the same rule
holds for every file that joblib or pickle wrote. Load only files you saved yourself or got from a
source you trust as much as your own code.

The notebook is finished with its files, so this cell removes the scratch folder:

In [19]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


## Recap

- Save a fitted pipeline with `joblib.dump` and load it with `joblib.load`
- Record the versions a model needs in a file beside it, and pin them where it is loaded
- Load a model in a separate program, and check that its predictions match
- Keep your own pipeline functions in a module, and save the whole pipeline, never the model alone
- Load only model files you made or files from a source you trust

## What is next

The **A Complete Workflow** notebook takes a raw table to a tested model and saves it this way, with
its versions beside it.

---

&#8592; **Previous:** [Mixed Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/12-mixed-column-types.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Complete Workflow](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/14-a-complete-workflow.ipynb) &#8594;
